# 🔍 Aula 19 — Interpretabilidade e Explicabilidade (XAI)

**Disciplina:** IA Aplicada à Engenharia Química  
**Dataset:** cstr_exotermico_xai.csv — rendimento de CSTR exotérmico

---

## O que vamos fazer

Abrir a "caixa-preta" do XGBoost: *por que o modelo fez esta predição?*

- Importância global (SHAP summary plot)
- Dependência por feature (dependence plot)
- Explicação local (waterfall plot)
- **Validar a consistência física** com a termodinâmica do reator


## Contexto físico (árbitro final)

Reator **exotérmico**: conversão máxima em T_ótimo (~105 °C).

- T abaixo do ótimo → reação lenta; T acima → equilíbrio desloca p/ reagentes
- Curva em **sino** de rendimento vs T
- Pressão alta favorece (fase gasosa, Le Chatelier)
- C_feed alta dilui; vazão alta reduz tempo de residência → conversão cai


## 3.1 — Exercício Guiado: SHAP no reator

Siga as células. Pipeline: treinar → explainer → summary → dependence → waterfall → validar fisicamente.


### Passo 1: treinar XGBoost


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula19/cstr_exotermico_xai.csv"
df = pd.read_csv(URL)

feats = ['T_reator_C','pressao_bar','C_feed_mol_L','vazao_L_min']
X, y = df[feats], df['rendimento_pct']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=42, verbosity=0)
xgb.fit(Xtr, ytr)
print("Modelo treinado")


### Passo 2: criar explainer + SHAP values


In [ ]:
import shap
explainer = shap.TreeExplainer(xgb)
shap_values = explainer(Xte)
print(shap_values.shape)


### Passo 3: summary plot (importância global + direção)


In [ ]:
shap.summary_plot(shap_values, Xte, max_display=4)
# Vermelho = valor alto da feature; Azul = baixo
# A direcao do SHAP mostra o sinal do efeito na predicao


### Passo 4: dependence plot de T (deve "dobrar")


In [ ]:
shap.dependence_plot('T_reator_C', shap_values.values, Xte)
# Fisica: SHAP + ate ~105 C, depois - (sino do equilibrio exotermico)


### Passo 5: dependence plot de P e vazão


In [ ]:
shap.dependence_plot('pressao_bar', shap_values.values, Xte)


### Passo 6: waterfall de 2 predições


In [ ]:
# Indice de uma predicao CERTA e uma ERRADA (erro)
yp = xgb.predict(Xte)
err = np.abs(yp - yte.values)
i_certa = int(np.argmin(err))
i_errada = int(np.argmax(err))
print(f"certa: erro={err[i_certa]:.3f}  errada: erro={err[i_errada]:.3f}")
shap.plots.waterfall(shap_values[i_certa])


### Passo 7: waterfall da predição errada + conclusão


In [ ]:
shap.plots.waterfall(shap_values[i_errada])
# O waterfall mostra qual feature afastou a predicao do valor real


> **Conclusão de consistência física:** o SHAP de T replica a curva em sino do
> equilíbrio exotérmico? Os sinais de P (+), C_feed (+) e vazão (−) concordam
> com a termodinâmica?


---

## 3.2 — Exercício em Grupo: Diagnóstico com Waterfall

Cada grupo investiga 1 predição SUSPEITA (erro > 5%). O waterfall revela a feature que causou o erro.

Checklist: [ ] waterfall gerado, [ ] feature dominante, [ ] sinal vs termodinâmica, [ ] hipótese (leakage/overfitting/colinearidade/interação), [ ] plano de correção.


## Checklist final

- [ ] XGBoost treinado
- [ ] explainer + shap_values
- [ ] summary plot
- [ ] dependence de T, P, vazão
- [ ] waterfall de 2 predições
- [ ] consistência física validada
- [ ] diagnóstico de inconsistências
